In [1]:
from opt_targeted_transfers import RateTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
tt = RateTargetedTransfers(c_bar=2.15, budget=None)
tt.fit(train_dataset=train_dataset, validation_dataset=validation_dataset)
tt.set_budget(0.5)

Fitting conditional densities vs glm spline method...


100%|██████████| 300/300 [00:11<00:00, 25.54it/s, loss=-0.0285, val_loss=-0.0799]


Final Theta: tensor([[-1.3137,  1.5011],
        [ 0.1991,  0.1164],
        [-0.0903, -0.4838],
        [-0.2246,  0.8079],
        [-0.6999,  0.0322],
        [-1.1252,  0.7672],
        [-2.0302,  2.0089]], dtype=torch.float64)


In [4]:
assignments = tt.run_opt(
   test_covariate_dataset, n_alpha=1, max_alpha=0.2, min_alpha=0.2, path="malawi_example_rate_B=0.5.csv"
)

Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:10<00:00, 10.31s/it]


In [5]:
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.2806746742793734,
 'post_transfer_poverty_rate': 0.37518383632166963,
 'policy_cost_per_capita': 0.4999999999999974,
 'policy_type': 'rate',
 'd': 2}

In [6]:
tt.set_budget(2.0)
tt.run_opt(
   test_covariate_dataset, n_alpha=1, max_alpha=0.2, min_alpha=0.2, path="malawi_example_rate_B=2.0.csv"
)
res = tt.evaluate(test_dataset)
res

Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.96s/it]


{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.0005068611012642895,
 'post_transfer_poverty_rate': 0.005568796402797536,
 'policy_cost_per_capita': 1.9999999999999827,
 'policy_type': 'rate',
 'd': 2}

In [7]:
tt.compute_auc(test_covariate_dataset=test_covariate_dataset, test_dataset=test_dataset, metrics=["post_transfer_poverty_rate",
                                                              "post_transfer_poverty_gap"], budgets=[0.05, 0.1, 0.5, 1.0, 2.0], min_alpha=0.2, max_alpha=0.2, n_alpha=1)

Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.71s/it]


Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.88s/it]


Alpha range: 0.2, 0.2


...

Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.99s/it]


Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.95s/it]


{'post_transfer_poverty_rate': {'auc': 0.4360746886905731,
  'results': [0.5571996262020763,
   0.5369756495486973,
   0.37518383632166963,
   0.1729440697878856,
   0.005568796402797536]},
 'post_transfer_poverty_gap': {'auc': 0.31822790955716285,
  'results': [0.4269822885783773,
   0.41072588698959883,
   0.2806746742793734,
   0.11811065839159134,
   0.0005068611012642895]}}